In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres.vectorstores import PGVector


load_dotenv()

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

connection_url = os.getenv("DATABASE_URL")

vector_store = PGVector(
  embeddings=embedding_model,
  connection=connection_url,
  collection_name="university_rules",
  use_jsonb=True # 메타데이터를 JSONB로 저장  
)

In [2]:
from langchain_core.documents import Document

documents_to_insert = [
    # 1번 문서: 2026학년도 도서관 대출 규정
    Document(
        page_content="[도서관 규정] 학부생은 최대 5권까지 14일간 도서 대출이 가능합니다.",
        metadata={
            "source": "library_guide_2026.pdf",
            "category": "library",
            "year": 2026
        }
    ),
    # 2번 문서: 2027학년도 개정 도서관 대출 규정
    Document(
        page_content="[개정 도서관 규정] 학부생은 최대 7권까지 21일간 도서 대출이 가능합니다.",
        metadata={
            "source": "library_guide_2027.pdf",
            "category": "library",
            "year": 2027
        }
    ),
    # 3번 문서: 2027학년도 장학금 규정
    Document(
        page_content="[장학금 규정] 직전 학기 평점 3.5 이상인 학생은 성적 우수 장학금 대상자입니다.",
        metadata={
            "source": "scholarship_2027.pdf",
            "category": "scholarship",
            "year": 2027
        }
    )
]

# 각 문서에 일대일로 매핑될 고유 식별자(ID)를 지정합니다.
document_ids = ["doc_lib_2026", "doc_lib_2027", "doc_sch_2027"]

saved_ids = vector_store.add_documents(
    documents=documents_to_insert,
    ids=document_ids
)

In [3]:
metadata_filter = {
    "year": 2027,
    "category": "library"
}

search_results = vector_store.similarity_search(
    query="최대 대출 일수를 알려주세요.",
    k=1,
    filter=metadata_filter
)

search_doc = search_results[0]
print(f"검색 본문: {search_doc.page_content}")
print(f"메타데이터: {search_doc.metadata}")

검색 본문: [개정 도서관 규정] 학부생은 최대 7권까지 21일간 도서 대출이 가능합니다.
메타데이터: {'year': 2027, 'source': 'library_guide_2027.pdf', 'category': 'library'}
